# ZeroDefect — Development Notebook

> **Academic prototype for surface defect detection using classical image processing.**
>
> Run cells top-to-bottom. Each section maps to one pipeline stage.

### How to use on Google Colab
1. Upload this notebook to Colab
2. Upload the `src/` folder to your Colab environment
3. Install requirements: `!pip install opencv-python scikit-image scikit-learn pandas scipy matplotlib seaborn plotly`
4. Upload dataset images to `/content/data/`
5. Run all cells

In [ ]:
# ── Install dependencies (run once) ─────────────────────────
# Uncomment if running on Colab:
# !pip install opencv-python scikit-image scikit-learn pandas scipy matplotlib seaborn plotly

import sys, os
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path().absolute().parent))

print('Environment ready.')

## Stage 1 & 2: Image Loading and Preprocessing

In [ ]:
from src.preprocessing import load_image, preprocess, get_image_info

# ── Load image ───────────────────────────────────────────────
IMAGE_PATH = '../data/leather/test/color/000.png'  # <── Change to your image path

# For demo without a real dataset, create a synthetic image:
np.random.seed(42)
demo = np.ones((256, 256, 3), dtype=np.uint8) * 170
demo[80:85, 20:230, :] = 30  # Horizontal scratch
demo += np.random.randint(0, 10, demo.shape, dtype=np.uint8)
image = demo

info = get_image_info(image)
print('Image info:', info)

plt.figure(figsize=(4, 4))
plt.imshow(image)
plt.title('Original Image')
plt.axis('off')
plt.tight_layout()
plt.savefig('../outputs/figures/original.png', dpi=100)
plt.show()
print('Saved: outputs/figures/original.png')

In [ ]:
# ── Preprocessing pipeline ───────────────────────────────────
prep = preprocess(image, blur_method='gaussian', blur_kernel=5, use_clahe=True, clahe_clip=2.0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (key, title) in zip(axes, [
    ('original', 'Original'), ('gray', 'Grayscale'),
    ('blurred', 'Gaussian Blur'), ('enhanced', 'CLAHE Enhanced')
]):
    cmap = 'gray' if prep[key].ndim == 2 else None
    ax.imshow(prep[key], cmap=cmap)
    ax.set_title(title, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.savefig('../outputs/figures/preprocessing.png', dpi=100)
plt.show()
print('Preprocessing stages complete.')

## Stage 3 & 4: Edge Detection and Thresholding

In [ ]:
from src.edge_detection import canny_edges, auto_canny, sobel_gradient
from src.segmentation import otsu_threshold, adaptive_threshold

enhanced = prep['enhanced']

edges_auto = auto_canny(enhanced)
edges_manual = canny_edges(enhanced, low_threshold=50, high_threshold=150)
grad = sobel_gradient(enhanced)
otsu_val, otsu_mask = otsu_threshold(enhanced)
adapt_mask = adaptive_threshold(enhanced, block_size=31, C=5)

print(f'Otsu threshold value: {otsu_val:.1f}')

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, img, title in zip(axes, 
    [enhanced, edges_auto, edges_manual, otsu_mask, adapt_mask],
    ['Enhanced', 'Auto-Canny', 'Manual Canny', 'Otsu Mask', 'Adaptive Mask']):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.savefig('../outputs/figures/edge_threshold.png', dpi=100)
plt.show()

## Stage 5: Morphological Processing

In [ ]:
from src.segmentation import opening, closing, erode, dilate, clean_mask

raw_mask = cv2.bitwise_or(edges_auto, otsu_mask)
opened = opening(raw_mask, kernel_size=3)
closed = closing(opened, kernel_size=7)
cleaned = clean_mask(raw_mask, open_kernel=3, close_kernel=7, min_area=100)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, title in zip(axes,
    [raw_mask, opened, closed, cleaned],
    ['Raw Combined', 'After Opening', 'After Closing', 'Final Cleaned']):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.savefig('../outputs/figures/morphology.png', dpi=100)
plt.show()
print(f'Flagged pixels: {np.count_nonzero(cleaned)} / {cleaned.size}')

## Stage 6: Texture Feature Extraction

In [ ]:
from src.texture_features import compute_glcm_features, compute_statistics, local_std_map, compute_lbp

glcm = compute_glcm_features(enhanced)
stats = compute_statistics(enhanced)
std_map = local_std_map(enhanced, window_size=15)
lbp_img = compute_lbp(enhanced)

print('GLCM Features:')
for k, v in glcm.items():
    print(f'  {k:15s}: {v:.4f}')
print('\nStatistics:')
for k, v in stats.items():
    print(f'  {k:15s}: {v:.2f}')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(enhanced, cmap='gray'); axes[0].set_title('Enhanced'); axes[0].axis('off')
std_disp = cv2.normalize(std_map, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
axes[1].imshow(std_disp, cmap='hot'); axes[1].set_title('Local Std-Dev Map'); axes[1].axis('off')
lbp_disp = cv2.normalize(lbp_img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
axes[2].imshow(lbp_disp, cmap='gray'); axes[2].set_title('LBP Image'); axes[2].axis('off')
plt.tight_layout()
plt.savefig('../outputs/figures/texture.png', dpi=100)
plt.show()

## Stage 7 & 8: Full Detection + Region Analysis

In [ ]:
from src.defect_detection import detect_defects_classical, create_overlay, analyze_regions
import pandas as pd

result = detect_defects_classical(image, min_defect_area=80)

print(f"Verdict       : {result['verdict']}")
print(f"Defect count  : {result['defect_count']}")
print(f"Area flagged  : {result['flagged_pct']:.2f}%")

overlay = create_overlay(image, result['clean_mask'])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, title in zip(axes,
    [image, result['clean_mask'], result['annotated'], overlay],
    ['Original', 'Defect Mask', 'Annotations', 'Overlay']):
    cmap = 'gray' if img.ndim == 2 else None
    ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.savefig('../outputs/figures/detection_result.png', dpi=100)
plt.show()

if result['regions']:
    df_regions = pd.DataFrame(result['regions'])
    df_regions.to_csv('../outputs/metrics/regions.csv', index=False)
    print('\nRegion data:')
    display(df_regions)

## Stage 9: Evaluation (if ground-truth mask available)

In [ ]:
from src.evaluation import pixel_metrics

# ── Synthetic ground truth for demo ─────────────────────────
# In real evaluation: load GT mask from MVTec AD ground_truth/ folder
gt_mask = np.zeros(image.shape[:2], dtype=np.uint8)
gt_mask[78:88, 15:235] = 255  # True scratch location

pred_mask = result['clean_mask']
metrics = pixel_metrics(pred_mask, gt_mask)

print('\n=== Pixel-Level Evaluation Metrics ===')
print(f"  Precision : {metrics['precision']:.4f}")
print(f"  Recall    : {metrics['recall']:.4f}")
print(f"  F1-Score  : {metrics['f1']:.4f}")
print(f"  IoU       : {metrics['iou']:.4f}")
print(f"  TP: {metrics['tp']}  FP: {metrics['fp']}  FN: {metrics['fn']}  TN: {metrics['tn']}")
print('\nNote: Results on synthetic demo image. Run on real MVTec data for valid evaluation.')

pd.DataFrame([metrics]).to_csv('../outputs/metrics/eval_metrics.csv', index=False)
print('Metrics saved to outputs/metrics/eval_metrics.csv')

## Texture Anomaly Detector (Approach B)

In [ ]:
from src.defect_detection import TextureAnomalyDetector

# For real use: replace with actual normal images
# detector.fit_from_directory('../data/leather/train/good/')

# Demo: use blurred version as pseudo-normal baseline
detector = TextureAnomalyDetector(z_threshold=2.0)
blurred_rgb = cv2.cvtColor(prep['blurred'], cv2.COLOR_GRAY2RGB)
detector.fit([blurred_rgb] * 8)  # 8 copies of "normal" blurred image

result_b = detector.predict(image, min_defect_area=80)

print(f"Verdict: {result_b['verdict']}")
print(f"Defects: {result_b['defect_count']}, Flagged: {result_b['flagged_pct']:.2f}%")

# Visualize anomaly score map
score = result_b['anomaly_score_map']
score_norm = cv2.normalize(score, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
overlay_b = create_overlay(image, result_b['clean_mask'])

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(score_norm, cmap='hot'); axes[1].set_title('Anomaly Score Map'); axes[1].axis('off')
axes[2].imshow(overlay_b); axes[2].set_title('Detection Overlay'); axes[2].axis('off')
plt.tight_layout()
plt.savefig('../outputs/figures/approach_b.png', dpi=100)
plt.show()